# Problem 1. 신용카드 이상 거래 탐지하기

## 문제 개요
카드사는 승인 요청이 들어온 거래 중 **부정 사용(사기) 거래**를 실시간으로 찾아내야 합니다.
거래 정보로 각 거래가 사기일 **확률**을 예측하세요.

## 파일 구조
```
Problem1/
├── problem1.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 70,000건 (Class 포함, 사기 358건)
- `data/test.csv` : 30,000건 (Class 없음)

| 컬럼 | 설명 |
|---|---|
| id | 거래 고유 번호 |
| Time | 데이터 수집 시작 이후 경과 시간(초), 약 이틀치 |
| V1 ~ V28 | 개인정보 보호를 위해 PCA로 변환된 거래 특성 (의미 비공개) |
| Amount | 거래 금액(유로) |
| **Class** | **1 = 사기, 0 = 정상 (예측 대상)** |

### 데이터 특징
- 사기 거래는 전체의 약 **0.5%** 로 극단적으로 불균형합니다.
- 출처: Credit Card Fraud Detection (ULB Machine Learning Group, ODbL)을 대회용으로 재구성. 정상 거래는 일부만 샘플링했습니다.

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem1/`)에 **`submission.csv`** 로 저장

## 평가
**Average Precision (PR 곡선 아래 면적)** — 높을수록 좋습니다.

- 제출한 값의 **순서**로 계산하는 지표입니다. 사기일수록 큰 값을 주면 됩니다.
- 0/1 라벨보다 **0~1 사이 확률(점수)** 을 제출하는 것이 유리합니다.

## 초기 코드 (베이스라인)
아래 베이스라인은 로지스틱 회귀로 0/1 라벨을 예측해 제출합니다. 동작은 하지만 점수가 낮습니다.

## 최종 풀이: 트리 3종 + 로지스틱 순위 평균
- 지표가 AP(순서 기반)이므로 확률 대신 점수의 **순위**를 평균해 제출합니다.
- 5-Fold OOF AP(실측): LR 0.839, HGB 0.882, RF 0.876, ExtraTrees 0.885, 4개 순위 평균 **0.891** (사기 358건뿐이라 ±0.01 정도는 노이즈 가능).
- 허용 라이브러리(sklearn)만 사용합니다. CatBoost 를 더한 버전은 0.893으로 차이가 노이즈 수준(부트스트랩 95% 구간이 0 포함)이라 쓰지 않았습니다.

In [1]:
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("data/train.csv"); test = pd.read_csv("data/test.csv")
feats = [c for c in train.columns if c not in ("id", "Class")]
X, y, Xt = train[feats].values, train.Class.values, test[feats].values

models = {
    "lr":  lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500)),
    "hgb": lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, random_state=0),
    "rf":  lambda: RandomForestClassifier(500, max_features=0.3, n_jobs=-1, random_state=0),
    "et":  lambda: ExtraTreesClassifier(500, max_features=0.5, n_jobs=-1, random_state=0),
}
# 5-Fold OOF 검증
oof = {k: np.zeros(len(y)) for k in models}
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
    for k, m in models.items(): oof[k][b] = m().fit(X[a], y[a]).predict_proba(X[b])[:, 1]
for k in models: print(k, "OOF AP: %.4f" % average_precision_score(y, oof[k]))
print("순위 평균 OOF AP: %.4f" % average_precision_score(y, sum(rankdata(v) for v in oof.values())))

# 전체 학습 후 test 예측 (순위 평균을 0~1 로 정규화)
pred = sum(rankdata(m().fit(X, y).predict_proba(Xt)[:, 1]) for m in models.values())
score = (pred - pred.min()) / (pred.max() - pred.min())
sub = pd.DataFrame({"id": test.id, "Class": score})
assert list(sub.columns) == ["id", "Class"] and len(sub) == len(test) and sub.Class.between(0, 1).all()
sub.to_csv("submission.csv", index=False)
sub.head()

lr OOF AP: 0.8391
hgb OOF AP: 0.8817
rf OOF AP: 0.8757
et OOF AP: 0.8852
순위 평균 OOF AP: 0.8912


,id,Class
0,70000,0.026726
1,70001,0.107330
2,70002,0.945411
3,70003,0.455448
4,70004,0.457894


## 제출 형식
`submission.csv`

| id | Class |
|---|---|
| 70000 | 0.0012 |
| 70001 | 0.9731 |

- 컬럼명 `id`, `Class`, 값은 실수(사기 확률 또는 점수)
- 포맷이 틀리면 0점 처리됩니다.

In [2]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()

(30000, 2)


,id,Class
0,70000,0.026726
1,70001,0.107330
2,70002,0.945411
3,70003,0.455448
4,70004,0.457894
